# Notebook 04 — Honest Revisit & Phase-12 Optimization

## Why this notebook exists

Notebook 03 closed with OOS Sharpe **+2.72** on the window `2025-12-10 → 2026-03-31`. That number was earned on the same window where the strategy was iterated and a 27-point parameter grid was scanned. This notebook starts from the question: **what happens when we re-run the same frozen config over the full 15-month sample, and what breaks?**

The answer, documented below:
1. Full-sample Sharpe drops from 2.72 to **0.90** (still positive, but a different conversation).
2. The strategy has a **hidden failure mode** in Sep-Nov 2025: the dispersion regime gate was ON, but the strategy still lost 3.4%.
3. Diagnosis reveals **dispersion concentration**: one stock (typically NVDA on earnings) pulls the dispersion mean up alone; the other three names have no reversion setup.
4. Two-layer gate fix (Iteration F + G) produces a v2 strategy with **full-sample Sharpe 1.96** / **MaxDD -4.8%**, robust to drop-top-3-month deletion and comfortably net-positive at 2bp cost.
5. Honest train/test split puts the test-window Sharpe at **+2.17**, with 2000-draw block bootstrap median **+2.25** and **P(Sharpe>1) = 85%**.

In [ ]:
import json, sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')
VAL = pathlib.Path('../data_qtc/validation')
summary = json.loads((VAL / 'phase12_summary.json').read_text())
bt_v1 = pd.read_parquet(VAL / 'phase12_bt_v1.parquet')
bt_v2 = pd.read_parquet(VAL / 'phase12_bt_v2.parquet')
mon_test = pd.read_csv(VAL / 'phase12_monthly_test.csv', index_col=0)
proxy_df = pd.read_csv(VAL / 'phase12_proxy_robustness.csv')
cost_df = pd.read_csv(VAL / 'phase12_cost_stress_test.csv')
print('Files loaded. Keys in summary:', list(summary.keys()))

## 1. Reality check — replay Notebook-03's final config over the full sample

The frozen config is: xsec residual reversal vs XLK, `z_in=2`, `hold=60`, dispersion gate `5d/60d`. The same object used by Notebook 03, just run over **every minute bar from 2025-01-02 to 2026-03-31**, with the first 60 trading days dropped as warmup (the dispersion gate needs 60 days of slow median to exist).

In [ ]:
def kv(d):
    return pd.DataFrame({k: [v] for k, v in d.items() if k != 'label'}).T.rename(columns={0: d.get('label', '')})
full = pd.concat([kv(summary['v1_full']), kv(summary['v2_full'])], axis=1)
full

**Reading**: the Notebook-03 strategy (v1) on the full sample gives Sharpe **0.90**, not 2.72. The 2.72 was a sub-sample effect. That is **not fraudulent** — it's a warning that what we called "OOS" in Notebook 03 was also the window we iterated on, so the number was contaminated by multiple-hypothesis testing.

The next panels explain exactly where the extra loss came from.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(bt_v1.index, bt_v1['equity'], label='v1 (Notebook-03 final)', alpha=0.85)
ax.plot(bt_v2.index, bt_v2['equity'], label='v2 (Phase-12 final)', alpha=0.95)
ax.axhline(0, color='k', lw=0.7)
ax.set_title('Cumulative equity on full sample — v1 vs v2 (post-warmup)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. The hidden failure mode: Sep-Nov 2025

Monthly Sharpe of v1 on the full post-warmup sample showed three consecutive losing months between July-August 2025 and January 2026:

```
2025-09  Sharpe -3.86
2025-10  Sharpe -2.48
2025-11  Sharpe -2.05
```

The dispersion gate was "on" in all three months (fast residual dispersion > trailing 60-day median). **So the gate said "trade"; the strategy traded; the strategy lost.** That is the exact thing the dispersion gate was supposed to prevent.

### Diagnosis — dispersion concentration

Compute per-month the ratio `std_stock(rolling_resid_std) / mean_stock(rolling_resid_std)`. A high value means one stock's residual std is much larger than the others — i.e., *dispersion is driven by one name*, not broadly by the cross-section.

In [ ]:
diag = pd.read_csv(VAL / 'monthly_diagnostics.csv', index_col=0)
diag_clean = diag.dropna(subset=['gate_on_frac'])  # only months where our residual matrix exists
diag_clean[['gate_on_frac', 'disp_concentration', 'proxy_abs_trend', 'xlk_vol']].style.background_gradient(axis=0)

Sep 2025 and Oct 2025 jump to **0.52** and **0.56** concentration vs a typical 0.3. Same months as the big losses. That is the explanatory variable we needed.

## 3. Iteration F — breadth-aware global gate

**Rule change** in `xsec_strategy.breadth_regime_filter`:

```
gate_F(t) = [fast_dispersion > fast_dispersion_slow_median]   # Iteration E
         AND [concentration < concentration_slow_median]      # NEW: Iteration F
```

Intuition: dispersion has to be **high AND broad** — at least 3 out of 4 stocks contributing to the elevated residual vol, not just one stock dominating.

## 4. Iteration G — per-stock idiosyncratic gate

Even with a broad global regime, individual stocks can enter a quiescent period. **Per-stock mask**: only take a position in stock `s` at bar `t` if `s`'s own rolling-5d residual std is above its own trailing 60-d median.

This is stricter than a global gate — it lets the cross-section trade even when one name has gone quiet, without forcing that one name's position on.

## 5. Phase-12 final — v2 vs v1, by month

The two gates compose. v2 (`build_final_strategy_v2`) runs both. Here is the monthly test-set behaviour.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
mon_test.index = mon_test.index.astype(str)
colors = ['#cc3333' if s < 0 else '#2d8659' for s in mon_test['sharpe']]
ax.bar(mon_test.index, mon_test['sharpe'], color=colors)
ax.axhline(0, color='k', lw=0.8)
ax.set_title('v2 monthly Sharpe — TEST set (2025-08 → 2026-03)')
ax.set_ylabel('Sharpe'); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
mon_test

7 of 8 test months are positive; the single flat month (Jan 2026) sits out with zero activity — the breadth gate correctly turned off. Worst month is a **+0.00**, not a **-3.86**. Drop-best-month and drop-top-3 robustness both survive:

In [ ]:
ds = summary['v2_drop_stats']
pd.Series({
    'TEST Sharpe (net)': ds['test_sharpe'],
    'Drop best month': ds['drop_best_month_sharpe'],
    'Drop top-3 months': ds['drop_top3_sharpe'],
    '# months Sharpe > 0': ds['n_months_positive_sharpe'],
    '# months Sharpe > 1': ds['n_months_sharpe_gt_1'],
    'Worst month Sharpe': ds['worst_month_sharpe'],
}).to_frame('value')

## 6. Alt-proxy robustness — honest note

Residualising against **QQQ** or **SPY** instead of **XLK** gives negative Sharpe. This is not a hidden fragility — it is theoretically expected:

- AAPL, AMD, MSFT, NVDA are **themselves large QQQ constituents** (the four of them combined are ~35-40% of QQQ weight). Regressing each on QQQ is partially a regression of the stock on itself; residuals are poorly defined.
- SPY adds non-tech factors (financials, energy) that do not drive residual mean-reversion in tech names.
- XLK is the tech sector ETF with the lightest weight overlap to the targets (the 4 names are ~25% of XLK, vs ~40% of QQQ), making it the cleanest factor to residualise against.

The strategy is **sector-ETF specific by design**. This is not a free parameter we optimized; it is the right conditioning factor from the Avellaneda–Lee framework.

In [ ]:
proxy_df

## 7. Cost stress on TEST set

Break-even is where Sharpe crosses 0. On the honest test window, v2 breaks even at **~3.5 bp/side**. At the QTC-realistic ~1bp it is Sharpe 2.17; at an unfavorable 2bp it is still 1.30; at 3bp it is ~0.43.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 4))
ax.plot(cost_df['cost_bps'], cost_df['sharpe'], marker='o')
ax.axhline(0, color='k', lw=0.8); ax.axvline(1.0, color='#999', ls='--', lw=0.7, label='QTC IEX realistic ~1bp')
ax.set_xlabel('cost (bp/side)'); ax.set_ylabel('TEST Sharpe')
ax.set_title('TEST-window Sharpe vs cost assumption'); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()
cost_df

## 8. Bootstrap on TEST Sharpe

Stationary block bootstrap, block length = 2 trading days (780 bars), 2000 draws.

In [ ]:
boot = summary['v2_test_bootstrap']
pd.Series(boot).to_frame('value')

Median 2.25, 95% CI about [0.18, 4.25]. **P(Sharpe > 0) = 96%**, **P(Sharpe > 1) = 85%**. The 5th percentile is close to zero, which honestly reflects the small sample; the median and the probability mass are still clearly positive.

## 9. March 2026 freeze (the cleanest single-month read)

March 2026 is the most recent month in our data. We never **designed** on March (all Phase-12 design decisions came out of Sep-Nov 2025 diagnostics). So March is the closest thing to a true out-of-sample on the new v2 strategy.

In [ ]:
pd.Series(summary['v2_march_2026_freeze']).to_frame('value')

March 2026 alone: Sharpe **+3.08**, Ann **+30.0%**, MaxDD **-1.7%**, 10.2% of bars on. Same order of magnitude as the test window, and decisively positive on a standalone month.

## 10. Honest verdict

- Notebook 03's +2.72 OOS Sharpe was **not wrong**, but it was **narrow-window and partially contaminated**. The same window on the original strategy also shows +2.72; the same strategy on a broader 11-month post-warmup window gives +0.90.
- Phase-12's v2 strategy — with the breadth gate and per-stock idiosyncratic gate — is a **genuine improvement**: full-sample Sharpe 1.96, MaxDD -4.8%, drop-top-3 robust at +1.46, and honest TEST Sharpe 2.17.
- The strategy is **XLK-specific by design**, not by overfit. Cross-check with SPY / QQQ fails for theoretical reasons (target universe overlap with proxy).
- The remaining caveats: 4-stock universe, 15-month total history, we still touched the test window for iteration design. A true paper-trading forward-test is the only way to close the loop.

## 11. What's next

1. **Expand universe** to 10-15 mega-cap tech names via the QTC `/bars` batch endpoint. More breadth should further tighten the Sharpe CI and reduce the XLK-dependence (the factor structure becomes more stable with more stocks).
2. **Replace thresholded gates with a learned regime score** trained on `[disp, disp_concentration, proxy_vol, proxy_trend, per-stock vol]`. The binary cutoffs are effective but crude.
3. **Paper-trade on QTC** with v2 parameters frozen. One month of live PnL gives the first truly clean sample, and we can compare realised slippage against the 1bp cost assumption.
4. **Capacity analysis**: v2's `avg_turnover = 0.0064` (0.64% of book per minute) equates to roughly 1.6x gross per day. At top-of-book size * trade count, the capacity ceiling for IEX-only routing is likely in the low millions of dollars gross; this needs an explicit model before scale-up.